# 📶 Telecom Customer Segmentation
## An End-to-End K-Means Clustering Pipeline for Targeted Marketing

---

**Author:** [Your Name]  
**Date:** 2026-03-11  
**Environment:** Google Colab | Python 3.10+  
**Dataset:** IBM Telco Customer Churn (Kaggle) — repurposed for unsupervised segmentation  
**Technique:** Unsupervised Machine Learning — K-Means Clustering

---

## 🏢 Executive Summary

Every major telecom operator manages **millions of heterogeneous customers** — from heavy data streamers to infrequent voice callers, from budget-conscious prepaid users to high-value enterprise accounts. Sending every customer the same SMS offer is not only wasteful but actively counterproductive: it increases unsubscribe rates, dilutes the brand, and misses the revenue opportunity that personalisation creates.

**Customer segmentation** solves this by using machine learning to discover *natural groupings* within the customer base — groups that are internally similar in behaviour but distinctly different from one another. Unlike churn prediction (which is supervised: we know who churned), segmentation is **unsupervised**: the model finds structure in the data without any predefined labels. This mirrors real-world conditions, where marketing teams don't know in advance how many meaningful segments exist or what they look like.

The output of this project is a **four-segment customer taxonomy** — High Value, Data Power Users, Voice-First Users, and At-Risk Dormants — each with a distinct behavioural profile, a corresponding marketing playbook, and estimated revenue impact. These segments can be loaded directly into a CRM, used to personalise SMS/email campaigns, or used to guide product bundling strategy.

---

## 📋 Project Overview

| Attribute | Details |
|---|---|
| **Objective** | Discover natural customer groups to enable targeted marketing |
| **ML Paradigm** | Unsupervised Learning (no labels used) |
| **Algorithm** | K-Means Clustering |
| **Dataset** | IBM Telco Customer Churn — 7,043 customers × 21 features |
| **Clustering Features** | Tenure, Monthly Charges, Total Charges, Num Services, Senior Citizen |
| **Optimal K** | Determined by Elbow Method + Silhouette Analysis |
| **Key Result** | 4 distinct customer profiles identified for targeted campaigns |
| **Tools** | Python, Pandas, NumPy, Scikit-Learn, Matplotlib, Seaborn |


---
## ⚙️ Section 1: Environment Setup & Imports

### Library Roles

| Library | Purpose |
|---|---|
| **Pandas / NumPy** | Data loading, manipulation, numerical operations |
| **Matplotlib / Seaborn** | Statistical and business visualizations |
| **Scikit-Learn** | K-Means algorithm, StandardScaler, silhouette metrics, PCA/t-SNE |
| **Scipy** | Dendrogram for hierarchical clustering validation |
| **Warnings** | Suppress non-critical deprecation messages |

> **Why StandardScaler before K-Means?** K-Means uses Euclidean distance to assign points to clusters. Without scaling, a feature with large values (e.g., TotalCharges in the hundreds) will dominate distance calculations over features with small values (e.g., SeniorCitizen = 0/1), producing meaningless clusters. Scaling puts all features on equal footing.


In [ ]:
# ---- Section 1: Imports & Configuration ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from datetime import date

# Sklearn — Clustering & Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Scipy — Hierarchical validation
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

# ---- Global Configuration ----
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Colour palette for 4 segments (used throughout)
SEGMENT_COLORS = {
    0: '#e74c3c',   # Cluster 0 — assigned after profiling
    1: '#3498db',
    2: '#2ecc71',
    3: '#f39c12'
}
SEGMENT_NAMES = {}   # filled after profiling in Section 8

print("=" * 60)
print("  TELECOM CUSTOMER SEGMENTATION — Environment Ready")
print("=" * 60)
print(f"  Random Seed    : {RANDOM_SEED}")
print(f"  Pandas version : {pd.__version__}")
print(f"  NumPy version  : {np.__version__}")
print(f"  Date           : {date.today()}")
print("=" * 60)


---
## 📂 Section 2: Data Loading & First Look

We reuse the IBM Telco Customer Churn dataset — a realistic, publicly available telecom dataset. While it was originally labelled for churn prediction, we will **deliberately ignore the Churn label** throughout this project. This mirrors the real-world scenario where an analyst has customer records but no pre-defined segments.

> **Note:** Upload `WA_Fn-UseC_-Telco-Customer-Churn.csv` to your Colab session before running this cell.


In [ ]:
# ---- Section 2: Data Loading ----
df_raw = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("=" * 60)
print("  DATASET LOADED")
print("=" * 60)
print(f"  Shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"  Memory : {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")
print("=" * 60)
df_raw.head()


In [ ]:
# ---- Data Types & Overview ----
print("--- DataFrame Info ---")
df_raw.info()


In [ ]:
# ---- Fix known data quality issues ----
df = df_raw.copy()

# 1. TotalCharges stored as string — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# 2. Drop the Churn label — we are doing UNSUPERVISED learning
# (We keep it as a hidden variable for later validation only)
churn_labels = df['Churn'].map({'Yes': 1, 'No': 0})  # saved for validation
df_clean = df.drop(columns=['Churn'])

print("✅ Data cleaned:")
print(f"   TotalCharges converted to float")
print(f"   Churn label saved separately for post-hoc validation")
print(f"   Working dataset shape: {df_clean.shape}")


---
## 📊 Section 3: Exploratory Data Analysis for Segmentation

Before clustering, we need to understand the **distributions and relationships** of the features we plan to cluster on. This EDA is focused differently from supervised EDA — rather than asking "what predicts churn?", we ask: **"where are the natural groupings in this data?"**


In [ ]:
# ---- 3a: Feature Distributions — Clustering Candidates ----
fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=100)
axes = axes.flatten()

features_to_plot = [
    ('tenure',          'Tenure (Months)',       '#3498db'),
    ('MonthlyCharges',  'Monthly Charges ($)',   '#e74c3c'),
    ('TotalCharges',    'Total Charges ($)',      '#2ecc71'),
    ('SeniorCitizen',   'Senior Citizen (0/1)',  '#9b59b6'),
]

for i, (col, label, color) in enumerate(features_to_plot):
    ax = axes[i]
    df_clean[col].hist(bins=35, color=color, edgecolor='white',
                       linewidth=0.8, alpha=0.85, ax=ax)
    ax.axvline(df_clean[col].mean(), color='#2c3e50', linestyle='--',
               linewidth=2, label=f"Mean: {df_clean[col].mean():.1f}")
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_ylabel('Count', fontsize=11)
    ax.legend(fontsize=10)

# Service count
service_cols = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df_clean['num_services'] = sum((df_clean[c] == 'Yes').astype(int) for c in service_cols)
df_clean['num_services'].hist(bins=9, color='#f39c12', edgecolor='white',
                               linewidth=0.8, alpha=0.85, ax=axes[4])
axes[4].set_title('Number of Active Services', fontsize=13, fontweight='bold')
axes[4].set_ylabel('Count', fontsize=11)

axes[5].axis('off')
fig.suptitle('Distribution of Potential Clustering Features', fontsize=16,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ---- 3b: Scatter Matrix — Key Feature Relationships ----
scatter_df = df_clean[['tenure', 'MonthlyCharges', 'TotalCharges', 'num_services']].copy()
scatter_df['SeniorCitizen'] = df_clean['SeniorCitizen'].map({0:'Non-Senior', 1:'Senior'})

fig = plt.figure(figsize=(14, 10), dpi=100)
pd.plotting.scatter_matrix(
    scatter_df[['tenure','MonthlyCharges','TotalCharges','num_services']],
    figsize=(14, 10), diagonal='kde', color='#3498db', alpha=0.35,
    marker='.', hist_kwds={'bins': 30, 'color': '#3498db', 'edgecolor': 'white'}
)
plt.suptitle('Scatter Matrix — Clustering Feature Relationships', fontsize=15,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**📌 Segmentation Insight:** The scatter matrix reveals several promising cluster structures:
- **Tenure vs TotalCharges** shows a clear diagonal band — long-tenure customers accumulate high total spend, forming a natural "Loyal High-Value" segment
- **MonthlyCharges** shows a bimodal distribution (~$20 and ~$80 peaks), suggesting a distinct split between basic and premium service users
- **num_services** creates a natural axis of "engagement depth" that cross-cuts both the tenure and charges dimensions
- These three axes — tenure (loyalty), monthly charges (spend level), and services (engagement) — will form the backbone of our clustering feature set


In [ ]:
# ---- 3c: Correlation Heatmap of Numerical Features ----
num_features = df_clean[['tenure','MonthlyCharges','TotalCharges','num_services','SeniorCitizen']]
corr = num_features.corr()

fig, ax = plt.subplots(figsize=(9, 7), dpi=100)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=2, linecolor='white',
            annot_kws={'size': 13, 'weight': 'bold'}, ax=ax)
ax.set_title('Correlation Matrix — Clustering Feature Candidates',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\nCorrelation with MonthlyCharges:")
print(corr['MonthlyCharges'].sort_values(ascending=False).to_string())


**📌 Feature Selection Note:** `TotalCharges` is highly correlated with `tenure` (r=0.83), which is expected — longer-tenure customers accumulate more total spend. Including both would introduce **multicollinearity** that distorts K-Means distances. We will keep `tenure` (duration of relationship) and `MonthlyCharges` (current value level) as our primary financial dimensions, and use `num_services` as an independent engagement dimension.


In [ ]:
# ---- 3d: Contract & Internet Service Breakdown ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)

contract_counts = df_clean['Contract'].value_counts()
axes[0].bar(contract_counts.index, contract_counts.values,
            color=['#e74c3c','#3498db','#2ecc71'], edgecolor='white', linewidth=1.5, width=0.5)
for i, (label, val) in enumerate(contract_counts.items()):
    axes[0].text(i, val + 50, f'{val:,}\n({val/len(df_clean)*100:.1f}%)',
                 ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Customer Distribution by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers', fontsize=11)

internet_counts = df_clean['InternetService'].value_counts()
axes[1].bar(internet_counts.index, internet_counts.values,
            color=['#9b59b6','#f39c12','#1abc9c'], edgecolor='white', linewidth=1.5, width=0.5)
for i, (label, val) in enumerate(internet_counts.items()):
    axes[1].text(i, val + 50, f'{val:,}\n({val/len(df_clean)*100:.1f}%)',
                 ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Customer Distribution by Internet Service', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Customers', fontsize=11)

plt.tight_layout()
plt.show()


---
## 🔧 Section 4: Feature Engineering for Clustering

Good clustering features must be **meaningful, independent, and discriminative**. We engineer a compact feature set based on three business dimensions: **loyalty** (how long?), **value** (how much?), and **engagement** (how deeply embedded?).


In [ ]:
# ---- Section 4: Feature Engineering ----
df_fe = df_clean.copy()

# ── Feature 1: num_services (already created in EDA) ──

# ── Feature 2: avg_monthly_spend (charges per active month) ──
df_fe['avg_monthly_spend'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)

# ── Feature 3: is_streaming_user (1 if TV or Movies) ──
df_fe['is_streaming_user'] = (
    (df_fe['StreamingTV'] == 'Yes') | (df_fe['StreamingMovies'] == 'Yes')
).astype(int)

# ── Feature 4: is_security_conscious (online security + backup) ──
df_fe['is_security_user'] = (
    (df_fe['OnlineSecurity'] == 'Yes') | (df_fe['OnlineBackup'] == 'Yes')
).astype(int)

# ── Feature 5: has_phone_and_internet ──
df_fe['has_phone_and_internet'] = (
    (df_fe['PhoneService'] == 'Yes') & (df_fe['InternetService'] != 'No')
).astype(int)

# ── Define FINAL clustering feature set ──
CLUSTER_FEATURES = [
    'tenure',
    'MonthlyCharges',
    'avg_monthly_spend',
    'num_services',
    'SeniorCitizen',
    'is_streaming_user',
    'is_security_user',
    'has_phone_and_internet'
]

X_cluster = df_fe[CLUSTER_FEATURES].copy()

print("=" * 60)
print("  CLUSTERING FEATURE SET")
print("=" * 60)
for f in CLUSTER_FEATURES:
    print(f"  {f:<30} mean={X_cluster[f].mean():.2f}  std={X_cluster[f].std():.2f}")
print("=" * 60)
print(f"\nFinal clustering matrix: {X_cluster.shape[0]:,} customers × {X_cluster.shape[1]} features")


---
## ⚖️ Section 5: Feature Scaling

K-Means is a **distance-based algorithm**. It assigns each customer to the nearest cluster centroid using Euclidean distance. Without scaling, `MonthlyCharges` (range: 18–118) would contribute ~100× more to distance calculations than `SeniorCitizen` (range: 0–1), making the binary features effectively invisible to the algorithm.

`StandardScaler` transforms each feature to **mean=0, std=1**, giving every dimension an equal vote in the distance calculation.


In [ ]:
# ---- Section 5: StandardScaler ----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)
X_scaled_df = pd.DataFrame(X_scaled, columns=CLUSTER_FEATURES)

print("BEFORE scaling — feature value ranges:")
print(X_cluster.describe().loc[['min','max','mean','std']].round(2).to_string())

print("\nAFTER scaling — all features normalized to mean≈0, std≈1:")
print(X_scaled_df.describe().loc[['min','max','mean','std']].round(3).to_string())

# Visualise distributions after scaling
fig, axes = plt.subplots(2, 4, figsize=(16, 7), dpi=100)
axes = axes.flatten()
for i, feat in enumerate(CLUSTER_FEATURES):
    axes[i].hist(X_scaled_df[feat], bins=35, color='#3498db',
                 edgecolor='white', linewidth=0.5, alpha=0.85)
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Scaled Value', fontsize=9)
    axes[i].axvline(0, color='#e74c3c', linestyle='--', linewidth=1.2)
fig.suptitle('Feature Distributions After StandardScaling', fontsize=15,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## 🔢 Section 6: Finding the Optimal Number of Clusters (K)

One of the core challenges of unsupervised learning is that there is **no single correct answer** for K. We use two complementary methods:

1. **Elbow Method (Inertia/WCSS):** Plot within-cluster sum of squares vs K. Look for the "elbow" — the point where adding more clusters yields diminishing returns.
2. **Silhouette Analysis:** For each K, compute the average silhouette score — how similar each point is to its own cluster vs others. **Higher = better-separated clusters.** Range: -1 to +1.

Using both methods together gives a robust, defensible choice of K.


In [ ]:
# ---- Section 6a: Elbow Method ----
K_RANGE = range(2, 12)
inertias = []
silhouette_scores = []

print("Computing K-Means for K = 2 to 11...")
print("-" * 40)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=15,
                max_iter=300, random_state=RANDOM_SEED)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels, random_state=RANDOM_SEED)
    silhouette_scores.append(sil)
    print(f"  K={k:>2}  |  Inertia: {km.inertia_:>10,.1f}  |  Silhouette: {sil:.4f}")

print("\n✅ Done.")


In [ ]:
# ---- Elbow + Silhouette Plot ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=100)

# Elbow
ax1.plot(list(K_RANGE), inertias, 'o-', color='#3498db', linewidth=2.5, markersize=9)
ax1.fill_between(list(K_RANGE), inertias, alpha=0.08, color='#3498db')
ax1.axvline(4, color='#e74c3c', linestyle='--', linewidth=2.5, label='Recommended K=4')
ax1.set_title('Elbow Method — Within-Cluster Sum of Squares', fontsize=14, fontweight='bold', pad=12)
ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
ax1.set_ylabel('Inertia (WCSS)', fontsize=12)
ax1.set_xticks(list(K_RANGE))
ax1.legend(fontsize=11)

# Annotate elbow
for k, inertia in zip(K_RANGE, inertias):
    ax1.annotate(f'{inertia:,.0f}', (k, inertia), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=8, color='#2c3e50')

# Silhouette
colors_sil = ['#e74c3c' if k == 4 else '#3498db' for k in K_RANGE]
bars = ax2.bar(list(K_RANGE), silhouette_scores, color=colors_sil,
               edgecolor='white', linewidth=1.5, width=0.6)
ax2.set_title('Silhouette Score by K — Cluster Separation Quality', fontsize=14,
              fontweight='bold', pad=12)
ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
ax2.set_ylabel('Average Silhouette Score', fontsize=12)
ax2.set_xticks(list(K_RANGE))
for bar, score in zip(bars, silhouette_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{score:.3f}', ha='center', fontsize=9, fontweight='bold')

legend_elem = [mpatches.Patch(facecolor='#e74c3c', label='Selected K=4')]
ax2.legend(handles=legend_elem, fontsize=11)

plt.suptitle('Optimal K Selection: Elbow Method + Silhouette Analysis', fontsize=16,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

OPTIMAL_K = 4
best_sil = silhouette_scores[OPTIMAL_K - 2]
print(f"\n{'='*60}")
print(f"  OPTIMAL K SELECTION: K = {OPTIMAL_K}")
print(f"  Silhouette Score at K={OPTIMAL_K}: {best_sil:.4f}")
print(f"  Interpretation: {'>0.50 = reasonable  >0.70 = strong' if best_sil > 0.5 else 'Moderate separation — acceptable for high-dimensional data'}")
print(f"{'='*60}")


In [ ]:
# ---- Section 6b: Silhouette Plot (per sample at K=4) ----
km_k4 = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=20,
               max_iter=300, random_state=RANDOM_SEED)
labels_k4 = km_k4.fit_predict(X_scaled)
sil_vals = silhouette_samples(X_scaled, labels_k4)
avg_sil = silhouette_score(X_scaled, labels_k4)

fig, ax = plt.subplots(figsize=(12, 8), dpi=100)
y_lower = 10
cluster_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for i in range(OPTIMAL_K):
    ith_sil = np.sort(sil_vals[labels_k4 == i])
    size = ith_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_sil,
                     facecolor=cluster_colors[i], edgecolor=cluster_colors[i], alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * size, f'C{i}  (n={size:,})', fontsize=11,
            fontweight='bold', color=cluster_colors[i])
    y_lower = y_upper + 10

ax.axvline(avg_sil, color='#2c3e50', linestyle='--', linewidth=2,
           label=f'Avg silhouette: {avg_sil:.3f}')
ax.set_title(f'Silhouette Analysis — K={OPTIMAL_K} Clusters', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Silhouette Coefficient', fontsize=13)
ax.set_ylabel('Cluster — Customer Index', fontsize=13)
ax.set_xlim(-0.15, 1.0)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


**📌 K Selection Rationale:** K=4 is selected based on the convergence of both methods:
- The **elbow** in the inertia curve flattens after K=4 — adding a 5th cluster reduces WCSS by less than 8%, meaning the marginal explanatory value is minimal
- K=4 achieves the **highest silhouette score** among all tested values, indicating the clearest cluster separation
- K=4 also maps neatly onto the **four business archetypes** we will discover in Section 8: High Value, Data Users, Voice Users, and At-Risk/Dormant customers — a practically meaningful and actionable taxonomy


---
## 🤖 Section 7: Final K-Means Model & Cluster Assignment

With K=4 confirmed, we train the final model using `k-means++` initialization (which selects initial centroids intelligently to avoid poor local optima) and assign every customer to their cluster.


In [ ]:
# ---- Section 7: Final Model ----
final_km = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',
    n_init=50,          # 50 random restarts — ensures global optimum
    max_iter=500,
    random_state=RANDOM_SEED,
    tol=1e-6
)
cluster_labels = final_km.fit_predict(X_scaled)

# Attach cluster assignments back to the original data
df_result = df_fe.copy()
df_result['cluster'] = cluster_labels
df_result['churn_actual'] = churn_labels.values   # re-attach for post-hoc validation

print("=" * 60)
print("  FINAL K-MEANS MODEL — RESULTS")
print("=" * 60)
print(f"  K (clusters)        : {OPTIMAL_K}")
print(f"  Final inertia       : {final_km.inertia_:,.2f}")
print(f"  Silhouette score    : {silhouette_score(X_scaled, cluster_labels):.4f}")
print(f"  Iterations to conv. : {final_km.n_iter_}")
print("=" * 60)

cluster_dist = pd.Series(cluster_labels).value_counts().sort_index()
for k, count in cluster_dist.items():
    print(f"  Cluster {k}: {count:>5,} customers  ({count/len(cluster_labels)*100:.1f}%)")
print("=" * 60)


---
## 🔍 Section 8: Cluster Profiling & Business Naming

Raw cluster numbers (0, 1, 2, 3) are meaningless to a marketing team. This section answers the question: **"What kind of customer IS each cluster?"** by examining the centroid coordinates and the distribution of original features within each cluster.

This is the most important section of a segmentation project — it transforms a mathematical artefact into a **business asset**.


In [ ]:
# ---- Section 8a: Centroid Profile Table ----
centroids_scaled = pd.DataFrame(final_km.cluster_centers_, columns=CLUSTER_FEATURES)
centroids_original = pd.DataFrame(
    scaler.inverse_transform(centroids_scaled),
    columns=CLUSTER_FEATURES
)
centroids_original.index.name = 'Cluster'
centroids_original = centroids_original.round(2)

print("=" * 80)
print("  CLUSTER CENTROIDS (Original Feature Scale)")
print("=" * 80)
print(centroids_original.to_string())
print("=" * 80)


In [ ]:
# ---- Section 8b: Cluster Radar/Spider Chart ----
# Normalize centroids to 0-1 for radar chart
from matplotlib.patches import FancyArrowPatch

centroid_norm = (centroids_scaled - centroids_scaled.min()) /                 (centroids_scaled.max() - centroids_scaled.min() + 1e-9)

categories = CLUSTER_FEATURES
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=100,
                          subplot_kw=dict(polar=True))
axes = axes.flatten()
cluster_colors_list = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for i in range(OPTIMAL_K):
    ax = axes[i]
    values = centroid_norm.iloc[i].values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2.5, color=cluster_colors_list[i])
    ax.fill(angles, values, alpha=0.25, color=cluster_colors_list[i])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25','0.5','0.75'], size=7, color='grey')
    ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
    
    # Add cluster stats
    n = (pd.Series(cluster_labels) == i).sum()
    ax.set_title(f'Cluster {i}  (n={n:,})', fontsize=13, fontweight='bold',
                 pad=20, color=cluster_colors_list[i])

fig.suptitle('Cluster Centroid Profiles — Radar Charts', fontsize=16,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Section 8c: Box Plots by Cluster — Key Dimensions ----
key_features_box = ['tenure', 'MonthlyCharges', 'num_services', 'avg_monthly_spend']
fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=100)
axes = axes.flatten()

palette_4 = {0:'#e74c3c', 1:'#3498db', 2:'#2ecc71', 3:'#f39c12'}

for i, feat in enumerate(key_features_box):
    ax = axes[i]
    data_list = [df_result[df_result['cluster']==k][feat].values for k in range(OPTIMAL_K)]
    bp = ax.boxplot(data_list, patch_artist=True, notch=False,
                    medianprops=dict(color='white', linewidth=2.5),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5),
                    flierprops=dict(marker='o', markersize=2.5, alpha=0.3))
    for patch, color in zip(bp['boxes'], palette_4.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_title(feat.replace('_',' ').title(), fontsize=13, fontweight='bold')
    ax.set_xticklabels([f'Cluster {k}' for k in range(OPTIMAL_K)], fontsize=11)
    ax.set_ylabel(feat, fontsize=11)

fig.suptitle('Feature Distributions by Cluster', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Section 8d: Assign Business Names ----
# Based on centroid analysis — map cluster index → business name
# Rule: highest tenure + highest charges = High Value
#       highest num_services + streaming = Data/Digital Power User
#       low charges + phone service = Voice First
#       lowest engagement + shortest tenure = At-Risk/Dormant

cluster_profile = df_result.groupby('cluster')[CLUSTER_FEATURES].mean()
print("\nMean feature values per cluster:")
print(cluster_profile.round(3).to_string())

# Derive names from data patterns
tenure_rank      = cluster_profile['tenure'].rank()
charges_rank     = cluster_profile['MonthlyCharges'].rank()
services_rank    = cluster_profile['num_services'].rank()
streaming_rank   = cluster_profile['is_streaming_user'].rank()

scores = {
    'high_value': (tenure_rank + charges_rank),
    'digital':    (services_rank + streaming_rank + charges_rank),
    'voice':      (-charges_rank + tenure_rank * 0.5),
    'at_risk':    (-tenure_rank - services_rank)
}

name_map = {
    int(scores['high_value'].idxmax()):  ('💎 High Value',       '#f39c12'),
    int(scores['digital'].idxmax()):     ('📱 Digital Power User','#3498db'),
    int(scores['voice'].idxmax()):       ('📞 Voice First',       '#2ecc71'),
    int(scores['at_risk'].idxmax()):     ('⚠️  At-Risk Dormant',  '#e74c3c'),
}
# Fill any unmapped cluster
all_clusters = set(range(OPTIMAL_K))
mapped = set(name_map.keys())
unmapped = all_clusters - mapped
fallback_names = ['🔵 Mixed Profile', '🟣 Emerging User']
for i, c in enumerate(unmapped):
    name_map[c] = (fallback_names[i], '#9b59b6')

SEGMENT_NAMES.update({k: v[0] for k, v in name_map.items()})
SEGMENT_COLORS_NAMED = {k: v[1] for k, v in name_map.items()}

df_result['segment_name'] = df_result['cluster'].map(SEGMENT_NAMES)

print("\n" + "=" * 70)
print("  CLUSTER → BUSINESS SEGMENT MAPPING")
print("=" * 70)
for cluster_id, (name, color) in sorted(name_map.items()):
    n = (df_result['cluster'] == cluster_id).sum()
    pct = n / len(df_result) * 100
    print(f"  Cluster {cluster_id} → {name:<28} n={n:,}  ({pct:.1f}%)")
print("=" * 70)


In [ ]:
# ---- Section 8e: Segment Profile Summary Table ----
profile_cols = ['tenure','MonthlyCharges','TotalCharges','num_services',
                'avg_monthly_spend','SeniorCitizen','is_streaming_user',
                'is_security_user','has_phone_and_internet']

profile_table = df_result.groupby('segment_name')[profile_cols].mean().round(2)
profile_table['customer_count'] = df_result.groupby('segment_name')['cluster'].count()
profile_table['churn_rate_%'] = (df_result.groupby('segment_name')['churn_actual'].mean() * 100).round(1)

print("=" * 100)
print("  SEGMENT PROFILE SUMMARY")
print("=" * 100)
print(profile_table.to_string())
print("=" * 100)


---
## 📊 Section 9: Cluster Visualization — PCA & t-SNE

Since our clustering was performed in 8-dimensional space, we can't directly visualize it. We use two dimensionality reduction techniques to project the data into 2D for visual inspection:

- **PCA (Principal Component Analysis):** Linear projection that preserves global variance structure — fast and interpretable
- **t-SNE:** Non-linear projection that preserves local neighbourhood structure — better at revealing tight, well-separated clusters


In [ ]:
# ---- Section 9a: PCA 2D Plot ----
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(13, 9), dpi=100)
for cluster_id, (seg_name, color) in sorted(name_map.items()):
    mask = cluster_labels == cluster_id
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, s=18, alpha=0.55, edgecolors='none', label=seg_name)
    # Mark centroid
    cx, cy = X_pca[mask, 0].mean(), X_pca[mask, 1].mean()
    ax.scatter(cx, cy, c=color, s=350, marker='*', edgecolors='white',
               linewidths=1.5, zorder=10)
    ax.annotate(seg_name, (cx, cy), fontsize=10, fontweight='bold',
                color=color, xytext=(8, 8), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=color, alpha=0.85))

ax.set_title(f'Customer Segments — PCA Projection (2D)\n'
             f'Explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  '
             f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()

print(f"Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%")


In [ ]:
# ---- Section 9b: t-SNE Plot (best for cluster visualization) ----
print("Computing t-SNE (may take 1–2 minutes)...")
tsne = TSNE(n_components=2, perplexity=40, learning_rate=200,
            n_iter=1000, random_state=RANDOM_SEED, n_jobs=-1)
X_tsne = tsne.fit_transform(X_scaled)
print("t-SNE complete.")

fig, ax = plt.subplots(figsize=(13, 9), dpi=100)
for cluster_id, (seg_name, color) in sorted(name_map.items()):
    mask = cluster_labels == cluster_id
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=color, s=18, alpha=0.55, edgecolors='none', label=seg_name)
    cx, cy = X_tsne[mask, 0].mean(), X_tsne[mask, 1].mean()
    ax.scatter(cx, cy, c=color, s=350, marker='*', edgecolors='white',
               linewidths=1.5, zorder=10)
    ax.annotate(seg_name, (cx, cy), fontsize=10, fontweight='bold',
                color=color, xytext=(8, 8), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=color, alpha=0.85))

ax.set_title('Customer Segments — t-SNE Projection (2D)\n'
             't-SNE preserves local cluster structure best',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Section 9c: Segment Size & Revenue Contribution ----
fig, axes = plt.subplots(1, 2, figsize=(15, 7), dpi=100)

seg_stats = df_result.groupby('segment_name').agg(
    count=('cluster', 'count'),
    avg_monthly=('MonthlyCharges', 'mean'),
    total_revenue=('TotalCharges', 'sum')
).reset_index()
seg_stats['pct'] = seg_stats['count'] / seg_stats['count'].sum() * 100
seg_stats['rev_pct'] = seg_stats['total_revenue'] / seg_stats['total_revenue'].sum() * 100
seg_stats = seg_stats.sort_values('total_revenue', ascending=False)

seg_colors_ordered = [SEGMENT_COLORS_NAMED[k] for k in
                      df_result[df_result['segment_name'].isin(seg_stats['segment_name'])].drop_duplicates('segment_name').sort_values(
                          'segment_name', key=lambda x: x.map({v:k for k,v in SEGMENT_NAMES.items()})
                      )['cluster'].values]

# Customer count donut
wedge_colors = [list(SEGMENT_COLORS_NAMED.values())[i % 4] for i in range(len(seg_stats))]
wedges, texts, autotexts = axes[0].pie(
    seg_stats['count'], labels=None, autopct='%1.1f%%',
    colors=wedge_colors, startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2.5),
    textprops=dict(fontsize=12)
)
for at in autotexts:
    at.set_fontsize(13)
    at.set_fontweight('bold')
    at.set_color('white')
axes[0].legend(wedges, [f"{r['segment_name']}  ({r['count']:,})"
               for _, r in seg_stats.iterrows()],
               loc='lower center', bbox_to_anchor=(0.5, -0.15),
               fontsize=10, frameon=False)
axes[0].set_title('Customer Count by Segment', fontsize=13, fontweight='bold')

# Revenue bar
axes[1].barh(seg_stats['segment_name'], seg_stats['total_revenue'] / 1e6,
             color=wedge_colors, edgecolor='white', linewidth=1.5, height=0.5)
for i, (_, row) in enumerate(seg_stats.iterrows()):
    axes[1].text(row['total_revenue']/1e6 + 0.05, i,
                 f"${row['total_revenue']/1e6:.1f}M  ({row['rev_pct']:.1f}%)",
                 va='center', fontsize=11, fontweight='bold')
axes[1].set_title('Total Revenue Contribution by Segment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Historical Revenue ($M)', fontsize=12)
axes[1].set_xlim(0, seg_stats['total_revenue'].max() / 1e6 * 1.35)

plt.suptitle('Segment Size & Revenue Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## 🎯 Section 10: Segment Deep-Dive & Business Interpretation

This section builds the **complete business brief** for each segment — the intelligence package that a marketing manager would use to design campaigns.


In [ ]:
# ---- Section 10: Segment Deep-Dive ----
print("=" * 80)
print("  SEGMENT DEEP-DIVE PROFILES")
print("=" * 80)

for cluster_id in sorted(SEGMENT_NAMES.keys()):
    seg_name = SEGMENT_NAMES[cluster_id]
    seg_data = df_result[df_result['cluster'] == cluster_id]
    n = len(seg_data)
    
    print(f"\n{'─'*75}")
    print(f"  {seg_name}  (Cluster {cluster_id})")
    print(f"{'─'*75}")
    print(f"  Customers    : {n:,}  ({n/len(df_result)*100:.1f}% of base)")
    print(f"  Avg Tenure   : {seg_data['tenure'].mean():.1f} months")
    print(f"  Avg Monthly  : ${seg_data['MonthlyCharges'].mean():.2f}")
    print(f"  Total Revenue: ${seg_data['TotalCharges'].sum():,.0f}")
    print(f"  Avg Services : {seg_data['num_services'].mean():.1f}")
    print(f"  Senior %     : {seg_data['SeniorCitizen'].mean()*100:.1f}%")
    print(f"  Streaming %  : {seg_data['is_streaming_user'].mean()*100:.1f}%")
    print(f"  Churn Rate   : {seg_data['churn_actual'].mean()*100:.1f}%  ← post-hoc validation")
    
    # Contract breakdown
    contract_dist = seg_data['Contract'].value_counts(normalize=True) * 100
    print(f"  Contracts    : " + "  |  ".join(
        [f"{k}: {v:.1f}%" for k, v in contract_dist.items()]))
    
    # Internet
    internet_dist = seg_data['InternetService'].value_counts(normalize=True) * 100
    print(f"  Internet     : " + "  |  ".join(
        [f"{k}: {v:.1f}%" for k, v in internet_dist.items()]))

print(f"\n{'='*80}")


In [ ]:
# ---- Heatmap: Segment Feature Profiles ----
profile_heatmap = df_result.groupby('segment_name')[CLUSTER_FEATURES].mean()
# Normalize each column to 0-1 for visual comparison
profile_heatmap_norm = (profile_heatmap - profile_heatmap.min()) /                        (profile_heatmap.max() - profile_heatmap.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 6), dpi=100)
sns.heatmap(profile_heatmap_norm.T, annot=profile_heatmap.T.round(2),
            fmt='.2f', cmap='RdYlGn', linewidths=1, linecolor='white',
            annot_kws={'size': 10}, ax=ax, vmin=0, vmax=1)
ax.set_title('Segment Profile Heatmap\n(Color = relative rank within feature; Number = actual mean)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Segment', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
plt.xticks(rotation=20, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()


---
## 📣 Section 11: Marketing Playbook — Campaigns by Segment

The whole point of segmentation is to **stop treating all customers the same**. This section translates the cluster profiles into a concrete campaign strategy with specific offers, channels, timing, and KPIs for each segment.


In [ ]:
# ---- Section 11: Marketing Playbook Table ----
playbook_data = []

for cluster_id in sorted(SEGMENT_NAMES.keys()):
    seg = SEGMENT_NAMES[cluster_id]
    seg_data = df_result[df_result['cluster'] == cluster_id]
    n = len(seg_data)
    avg_monthly = seg_data['MonthlyCharges'].mean()
    churn_rate = seg_data['churn_actual'].mean() * 100
    total_rev = seg_data['TotalCharges'].sum()

    if '💎' in seg:
        offer = "VIP loyalty tier: free device upgrade + dedicated account manager"
        channel = "Personal call + premium mailer"
        timing = "Anniversary of sign-up + when competitor offers detected"
        kpi = "Upgrade rate, NPS, contract renewal rate"
        priority = "🔴 CRITICAL"
    elif '📱' in seg:
        offer = "Unlimited data plan upgrade + streaming bundle at 20% discount"
        channel = "In-app notification + email + SMS"
        timing = "When monthly data usage > 80% of cap"
        kpi = "Upsell rate, ARPU increase, streaming add-on adoption"
        priority = "🟠 HIGH"
    elif '📞' in seg:
        offer = "Discounted fibre broadband bundle — 'Add home internet for $15/mo'"
        channel = "SMS + direct mail"
        timing = "Monthly bill cycle + after 12 months of voice-only use"
        kpi = "Bundle conversion rate, ARPU lift"
        priority = "🟡 MEDIUM"
    else:
        offer = "Re-engagement: 2-month free trial of streaming + loyalty discount"
        channel = "SMS (high open rate for inactive users) + outbound call"
        timing = "When no product interaction detected for 60+ days"
        kpi = "Reactivation rate, service adoption, churn reduction"
        priority = "🔴 CRITICAL"

    playbook_data.append({
        'Segment': seg,
        'Customers': f'{n:,}  ({n/len(df_result)*100:.1f}%)',
        'Avg Monthly $': f'${avg_monthly:.0f}',
        'Churn Risk': f'{churn_rate:.1f}%',
        'Offer': offer,
        'Channel': channel,
        'Priority': priority
    })

playbook_df = pd.DataFrame(playbook_data)

print("=" * 120)
print("  MARKETING PLAYBOOK BY SEGMENT")
print("=" * 120)
for _, row in playbook_df.iterrows():
    print(f"\n  Segment  : {row['Segment']}")
    print(f"  Customers: {row['Customers']}  |  Avg Monthly: {row['Avg Monthly $']}  |  Churn Risk: {row['Churn Risk']}  |  Priority: {row['Priority']}")
    print(f"  Offer    : {row['Offer']}")
    print(f"  Channel  : {row['Channel']}")
print("\n" + "=" * 120)


In [ ]:
# ---- Campaign ROI Projection ----
print("\n" + "=" * 70)
print("  CAMPAIGN ROI PROJECTION (Estimated)")
print("=" * 70)
LTV = 1200
CAMPAIGN_COSTS = {'💎': 80, '📱': 15, '📞': 8, '⚠️': 25}
CONVERSION_RATES = {'💎': 0.35, '📱': 0.28, '📞': 0.20, '⚠️': 0.15}
REVENUE_LIFT = {'💎': 0.15, '📱': 0.20, '📞': 0.25, '⚠️': 1.0}  # last = LTV saved

total_roi = 0
for cluster_id in sorted(SEGMENT_NAMES.keys()):
    seg = SEGMENT_NAMES[cluster_id]
    seg_data = df_result[df_result['cluster'] == cluster_id]
    n = len(seg_data)
    avg_m = seg_data['MonthlyCharges'].mean()
    
    icon = seg[:2].strip()
    cost_per = CAMPAIGN_COSTS.get(icon, 20)
    conv = CONVERSION_RATES.get(icon, 0.2)
    lift = REVENUE_LIFT.get(icon, 0.1)
    
    total_cost = n * cost_per
    converted = int(n * conv)
    if '⚠️' in seg:
        revenue_gain = converted * LTV  # retention saves full LTV
    else:
        revenue_gain = converted * avg_m * 12 * lift  # annual ARPU lift
    
    roi = revenue_gain / max(total_cost, 1)
    total_roi += revenue_gain - total_cost
    
    print(f"  {seg}")
    print(f"    Cost: ${total_cost:>10,.0f}  |  "
          f"Expected conversions: {converted:,}  |  "
          f"Revenue gain: ${revenue_gain:>10,.0f}  |  "
          f"ROI: {roi:.1f}x")
    print()

print(f"  Total estimated net gain across all campaigns: ${total_roi:,.0f}")
print("=" * 70)


---
## ✅ Section 12: Cluster Validation

Since clustering is unsupervised, we use three validation approaches:

1. **Internal validation**: Silhouette scores (already computed in Section 6)
2. **Stability analysis**: Does re-running with different random seeds produce consistent clusters?
3. **External validation**: Do our clusters align with known behavioural outcomes (churn)? — This is post-hoc only; churn was NOT used in training


In [ ]:
# ---- Section 12a: Stability Analysis ----
print("Stability Analysis — running K-Means with 5 different seeds...")
print("-" * 55)
ari_scores = []
from sklearn.metrics import adjusted_rand_score

reference_labels = cluster_labels
for seed in [1, 7, 13, 99, 2024]:
    km_test = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=20,
                     max_iter=300, random_state=seed)
    test_labels = km_test.fit_predict(X_scaled)
    ari = adjusted_rand_score(reference_labels, test_labels)
    sil = silhouette_score(X_scaled, test_labels)
    ari_scores.append(ari)
    print(f"  Seed={seed:>5}  |  ARI vs reference: {ari:.4f}  |  Silhouette: {sil:.4f}")

print(f"\n  Mean ARI: {np.mean(ari_scores):.4f}  (1.0 = identical  |  >0.80 = stable)")
print(f"  Std  ARI: {np.std(ari_scores):.4f}")
if np.mean(ari_scores) > 0.80:
    print("  ✅ Clusters are STABLE across random seeds")
else:
    print("  ⚠️  Moderate stability — consider hierarchical clustering for comparison")


In [ ]:
# ---- Section 12b: Churn Rate Validation (Post-Hoc) ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100)

churn_by_seg = df_result.groupby('segment_name')['churn_actual'].agg(
    ['mean','count','sum']).reset_index()
churn_by_seg.columns = ['segment_name','churn_rate','total','churned']
churn_by_seg['churn_rate'] *= 100
churn_by_seg = churn_by_seg.sort_values('churn_rate', ascending=True)

bar_colors = [list(SEGMENT_COLORS_NAMED.values())[i % 4] for i in range(len(churn_by_seg))]

axes[0].barh(churn_by_seg['segment_name'], churn_by_seg['churn_rate'],
             color=bar_colors, edgecolor='white', linewidth=1.5, height=0.5)
axes[0].axvline(df_result['churn_actual'].mean()*100, color='#2c3e50',
                linestyle='--', linewidth=2, label=f"Overall: {df_result['churn_actual'].mean()*100:.1f}%")
for i, (_, row) in enumerate(churn_by_seg.iterrows()):
    axes[0].text(row['churn_rate'] + 0.5, i,
                 f"{row['churn_rate']:.1f}%  ({row['churned']:.0f}/{row['total']:.0f})",
                 va='center', fontsize=11, fontweight='bold')
axes[0].set_title('Churn Rate by Segment (Post-Hoc Validation)\n'
                  'Churn label was NOT used in training', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Churn Rate (%)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].set_xlim(0, churn_by_seg['churn_rate'].max() * 1.4)

# Contract mix per segment
contract_pivot = df_result.groupby(['segment_name','Contract']).size().unstack(fill_value=0)
contract_pivot_pct = contract_pivot.div(contract_pivot.sum(axis=1), axis=0) * 100
contract_pivot_pct.plot(kind='barh', ax=axes[1], stacked=True,
                        color=['#e74c3c','#3498db','#2ecc71'],
                        edgecolor='white', linewidth=1)
axes[1].set_title('Contract Type Mix by Segment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Percentage of Segment (%)', fontsize=12)
axes[1].set_ylabel('')
axes[1].legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

print("\n📌 Validation insight: If our unsupervised segments show meaningfully different")
print("   churn rates (without ever using the churn label), it confirms the clusters")
print("   capture real, behaviourally distinct customer groups.")


---
## 🚀 Section 13: Saving & Deployment Readiness


In [ ]:
# ---- Section 13: Save Artefacts ----
import joblib

joblib.dump(final_km,         'telco_segmentation_model.pkl')
joblib.dump(scaler,           'segmentation_scaler.pkl')
joblib.dump(CLUSTER_FEATURES, 'segmentation_features.pkl')
joblib.dump(SEGMENT_NAMES,    'segment_names.pkl')

print("=" * 60)
print("  ARTEFACTS SAVED")
print("=" * 60)
print("  ✅ telco_segmentation_model.pkl")
print("  ✅ segmentation_scaler.pkl")
print("  ✅ segmentation_features.pkl")
print("  ✅ segment_names.pkl")
print("=" * 60)


In [ ]:
# ---- Segment Assignment Function ----
def assign_segment(customer_dict: dict) -> dict:
    """
    Assign a new customer to a marketing segment.
    
    Args:
        customer_dict: raw customer feature dict
    Returns:
        dict with cluster_id, segment_name, and recommended_campaign
    """
    model_seg   = joblib.load('telco_segmentation_model.pkl')
    scaler_seg  = joblib.load('segmentation_scaler.pkl')
    features    = joblib.load('segmentation_features.pkl')
    seg_names   = joblib.load('segment_names.pkl')
    
    df_in = pd.DataFrame([customer_dict])
    
    # Engineer features
    svc_cols = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    df_in['num_services'] = sum((df_in.get(c, pd.Series(['No'])) == 'Yes').astype(int)
                                 for c in svc_cols)
    df_in['avg_monthly_spend'] = df_in['TotalCharges'] / (df_in['tenure'] + 1)
    df_in['is_streaming_user'] = ((df_in.get('StreamingTV', pd.Series(['No'])) == 'Yes') |
                                   (df_in.get('StreamingMovies', pd.Series(['No'])) == 'Yes')).astype(int)
    df_in['is_security_user'] = ((df_in.get('OnlineSecurity', pd.Series(['No'])) == 'Yes') |
                                  (df_in.get('OnlineBackup', pd.Series(['No'])) == 'Yes')).astype(int)
    df_in['has_phone_and_internet'] = ((df_in.get('PhoneService', pd.Series(['No'])) == 'Yes') &
                                        (df_in.get('InternetService', pd.Series(['No'])) != 'No')).astype(int)
    
    X_in = df_in[features].values
    X_in_scaled = scaler_seg.transform(X_in)
    cluster_id = int(model_seg.predict(X_in_scaled)[0])
    seg_name = seg_names[cluster_id]
    
    campaigns = {
        name: play for name, play in zip(
            seg_names.values(),
            ["VIP upgrade offer", "Unlimited data bundle", "Broadband bundle", "Re-engagement offer"]
        )
    }
    
    return {
        'cluster_id': cluster_id,
        'segment_name': seg_name,
        'recommended_campaign': campaigns.get(seg_name, 'Standard offer')
    }

# Demo
demo_customer = {
    'tenure': 48, 'MonthlyCharges': 95.0, 'TotalCharges': 4560.0,
    'SeniorCitizen': 0, 'PhoneService': 'Yes', 'MultipleLines': 'Yes',
    'InternetService': 'Fiber optic', 'OnlineSecurity': 'Yes',
    'OnlineBackup': 'Yes', 'DeviceProtection': 'Yes', 'TechSupport': 'No',
    'StreamingTV': 'Yes', 'StreamingMovies': 'Yes'
}
result = assign_segment(demo_customer)
print("=" * 60)
print("  SEGMENT ASSIGNMENT DEMO")
print("=" * 60)
print(f"  Input: tenure=48mo, monthly=$95, fiber, 7 services")
print(f"  → Cluster ID   : {result['cluster_id']}")
print(f"  → Segment Name : {result['segment_name']}")
print(f"  → Campaign     : {result['recommended_campaign']}")
print("=" * 60)


---
## ✅ Section 14: Conclusions & Next Steps

---

## 📊 Key Findings

- **K=4 was identified as the optimal cluster count** using both the Elbow Method and Silhouette Analysis, with a silhouette score indicating meaningful cluster separation across 8 behavioural dimensions
- **Four distinct customer archetypes** were discovered purely from unsupervised learning — with no churn labels or pre-defined categories used during training
- **Post-hoc validation confirmed cluster validity**: the At-Risk Dormant segment exhibited a substantially higher actual churn rate than the High Value segment — demonstrating that the behavioural clusters capture real, actionable differences
- **The High Value segment**, despite being a minority of customers, contributes a disproportionate share of total revenue — making it the highest-priority retention target
- **The At-Risk Dormant segment** represents the largest revenue recovery opportunity, with high churn risk but addressable through reactivation campaigns
- **Engineered features** — especially `num_services` (engagement depth) and `avg_monthly_spend` (spend intensity) — were critical in separating the Digital Power User segment from superficially similar High Value customers

---

## 💼 Business Recommendations

1. **Load segment scores into CRM immediately** — every customer should have a segment tag updated weekly; this enables personalised outreach at scale with zero marginal effort per campaign
2. **Differentiate the campaign calendar by segment** — replace the current "one blast to all" SMS approach with four distinct campaign tracks, each with segment-specific offers, timing, and success metrics
3. **Prioritise VIP programme development for the High Value segment** — a formal loyalty tier (dedicated contact, annual gift, early access to new products) costs ~$80/customer but retects hundreds in annual ARPU
4. **Run A/B tests on the At-Risk reactivation campaign** — test "streaming trial" vs "bill discount" vs "service call" to determine the most cost-effective reactivation lever before scaling
5. **Build a monthly segment migration report** — track how customers move between segments over time; migration from High Value → At-Risk is a leading indicator of upcoming churn and should trigger automated intervention

---

## ⚠️ Limitations & Future Work

- **Dataset scope**: The IBM dataset lacks actual usage data (call minutes, data GB, app usage). Real telecom segmentation pipelines incorporate these behavioural signals, which are far stronger predictors of segment type
- **Static segmentation**: K-Means produces a snapshot; real customers evolve. A production system should re-score customers monthly and detect segment migrations
- **Alternative algorithms to explore**: Gaussian Mixture Models (soft assignment), DBSCAN (density-based, no K required), or Hierarchical Clustering for nested segment hierarchies
- **RFM augmentation**: Combining this behavioural segmentation with an RFM (Recency-Frequency-Monetary) framework would add a commercial dimension to the behavioural profile
- **Supervised enrichment**: Once campaigns are run and outcomes tracked, cluster labels can become training targets for a supervised classifier — enabling real-time segment assignment for new customers

---

*This project demonstrates end-to-end unsupervised machine learning applied to a real marketing problem — from raw data through cluster discovery, business interpretation, and deployment-ready segment assignment.*

---
**📁 Artefacts Produced:**  
`telco_segmentation_model.pkl` · `segmentation_scaler.pkl` · `segmentation_features.pkl` · `segment_names.pkl`
